### 1. Preprocessing Text + Custom Tokenizer Emoji

In [1]:
import os
import re
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print('All imports OK.')

All imports OK.


In [2]:
MODEL_NAME = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

raw_emoji_to_tag = {
    '\U0001f61b': '<Wajah Menjulurkan Lidah>',
    '\U0001f620': '<Wajah Marah>',
    '\U0001f4a3': '<Bom>',
    '\U0001f494': '<Hati Patah>',
    '\U0001f615': '<Wajah Bingung>',
    '\U0001f61e': '<Wajah Kecewa>',
    '\U0001f611': '<Wajah Tanpa Ekspresi>',
    '\U0001f60b': '<Wajah Menikmati Makanan Lezat>',
    '\U0001f631': '<Wajah Berteriak Ketakutan>',
    '\U0001f613': '<Wajah Dengan Keringat Dingin>',
    '\U0001f62e': '<Wajah Terperangah Terbuka>',
    '\U0001f624': '<Wajah Mengeluarkan Uap Dari Hidung>',
    '\U0001f61d': '<Wajah Menjulurkan Lidah Dan Mata Tertutup>',
    '\U0001f636': '<Wajah Tanpa Mulut>',
    '\U0001f525': '<Api>',
    '\u2639': '<Wajah Merengut>',
    '\U0001f62c': '<Wajah Meringis>',
    '\u26a1': '<Tegangan Tinggi>',
    '\U0001f925': '<Wajah Berbohong>',
    '\U0001f623': '<Wajah Bertahan Menderita>',
    '\U0001f647': '<Orang Membungkuk>',
    '\U0001f3c3': '<Orang Berlari>',
    '\U0001f43d': '<Hidung Babi>',
    '\U0001f621': '<Wajah Bersungut Marah>',
    '\U0001f648': '<Monyet Menutup Mata>',
    '\U0001f641': '<Wajah Agak Merengut>',
    '\U0001f64a': '<Monyet Menutup Mulut>',
    '\U0001f914': '<Wajah Berpikir>',
    '\U0001f44e': '<Jempol Ke Bawah>',
    '\U0001f445': '<Lidah>',
    '\U0001f629': '<Wajah Lelah Lesu>',
    '\U0001f910': '<Wajah Mulut Resleting>',
    '\U0001f610': '<Wajah Netral>',
    '\U0001f644': '<Wajah Memutar Mata>',
    '\U0001f60f': '<Wajah Tersenyum Sinis>',
    '\U0001f625': '<Wajah Kecewa Tapi Lega>',
    '\U0001f62f': '<Wajah Terbungkam>',
    '\U0001f62a': '<Wajah Mengantuk>',
    '\U0001f62b': '<Wajah Sangat Lelah>',
    '\U0001f634': '<Wajah Tidur>',
    '\U0001f60c': '<Wajah Lega>',
    '\U0001f61c': '<Wajah Menjulurkan Lidah Dan Mengedipkan Mata>',
    '\U0001f924': '<Wajah Mengiler>',
    '\U0001f612': '<Wajah Tidak Terhibur>',
    '\U0001f614': '<Wajah Termenung Sedih>',
    '\U0001f643': '<Wajah Terbalik>',
    '\U0001f911': '<Wajah Mata Duitan>',
    '\U0001f632': '<Wajah Terperanjat>',
    '\U0001f626': '<Wajah Jengkel Bingung>',
    '\U0001f61f': '<Wajah Cemas>',
    '\U0001f622': '<Wajah Menangis>',
    '\U0001f62d': '<Wajah Menangis Kencang>',
    '\U0001f626': '<Wajah Merengut Dengan Mulut Terbuka>',
    '\U0001f627': '<Wajah Menderita>',
    '\U0001f628': '<Wajah Takut>',
    '\U0001f630': '<Wajah Mulut Terbuka Dan Keringat Dingin>',
    '\U0001f633': '<Wajah Memerah Tersipu>',
    '\U0001f635': '<Wajah Pusing>',
    '\U0001f637': '<Wajah Masker Medis>',
    '\U0001f912': '<Wajah Dengan Termometer>',
    '\U0001f915': '<Wajah Dengan Perban Di Kepala>',
    '\U0001f922': '<Wajah Mual>',
    '\U0001f927': '<Wajah Bersin>',
    '\U0001f913': '<Wajah Kutu Buku>',
    '\U0001f608': '<Wajah Tersenyum Berandalan>',
    '\U0001f47f': '<Wajah Marah Bertanduk>',
    '\U0001f479': '<Raksasa>',
    '\U0001f47a': '<Jin>',
    '\U0001f480': '<Tengkorak>',
    '\u2620': '<Tengkorak Dan Tulang Silang>',
    '\U0001f47b': '<Hantu>',
    '\U0001f4a9': '<Kotoran>',
    '\U0001f640': '<Wajah Kucing Letih>',
    '\U0001f63f': '<Wajah Kucing Menangis>',
    '\U0001f63e': '<Wajah Kucing Bersungut>',
    '\U0001f649': '<Monyet Menutup Telinga>',
    '\U0001f64e': '<Orang Bersungut>',
    '\U0001f645': '<Orang Isyarat Tidak>',
    '\U0001f481': '<Orang Menengadahkan Tangan>',
    '\U0001f926': '<Orang Tepok Jidat>',
    '\U0001f937': '<Orang Mengangkat Bahu>',
    '\U0001f91e': '<Jari Menyilang>',
    '\U0001f4c9': '<Grafik Menurun>',
    '\u26d4': '<Dilarang Masuk>',
    '\u2716': '<Tanda Silang Perkalian>',
    '\u274c': '<Tanda Silang>',
    '\u274e': '<Tombol Tanda Silang>',
    '\U0001f44c': '<Tangan Isyarat Ok>',
    '\U0001f44a': '<Tinjuan Ke Depan>',
    '\U0001f918': '<Isyarat Metal>',
    '\U0001f60d': '<Wajah Tersenyum Dengan Mata Hati>',
    '\U0001f60a': '<Wajah Tersenyum Dengan Mata Tersenyum>',
    '\U0001f44d': '<Jempol Ke Atas>',
    '\U0001f639': '<Wajah Kucing Menangis Bahagia>',
    '\U0001f44f': '<Tepuk Tangan>',
    '\U0001f618': '<Wajah Meniup Ciuman>',
    '\U0001f602': '<Wajah Menangis Gembira>',
    '\U0001f64f': '<Tangan Merapat Berdoa>',
    '\u270a': '<Kepalan Tangan Kejayaan>',
    '\U0001f31f': '<Bintang Bersinar>',
    '\U0001f601': '<Wajah Menyeringai Dengan Mata Tersenyum>',
    '\U0001f600': '<Wajah Menyeringai>',
    '\U0001f498': '<Hati Dengan Panah>',
    '\u2714': '<Tanda Centang Tebal>',
    '\U0001f917': '<Wajah Memeluk>',
    '\U0001f61a': '<Wajah Mencium Dengan Mata Tertutup>',
    '\u2764': '<Hati Merah>',
    '\U0001f64b': '<Orang Mengangkat Tangan>',
    '\U0001f64c': '<Mengangkat Kedua Tangan>',
    '\U0001f923': '<Tertawa Terpingkal-Pingkal>',
    '\U0001f606': '<Wajah Tersenyum Mulut Terbuka Dan Mata Tertutup>',
    '\U0001f605': '<Wajah Tersenyum Mulut Terbuka Dan Keringat Dingin>',
    '\U0001f604': '<Wajah Tersenyum Mulut Terbuka Dan Mata Tersenyum>',
    '\U0001f60e': '<Wajah Tersenyum Dengan Kacamata Hitam>',
    '\U0001f3c6': '<Piala>',
    '\u270c': '<Tangan Tanda Damai Berjaya>',
    '\U0001f603': '<Wajah Tersenyum Dengan Mulut Terbuka>',
    '\U0001f609': '<Wajah Mengedipkan Mata>',
    '\U0001f617': '<Wajah Mencium>',
    '\U0001f619': '<Wajah Mencium Dengan Mata Tersenyum>',
    '\u263a': '<Wajah Tersenyum Simpel>',
    '\U0001f642': '<Wajah Agak Tersenyum>',
    '\U0001f607': '<Wajah Tersenyum Malaikat>',
    '\U0001f920': '<Wajah Topi Koboi>',
    '\U0001f921': '<Wajah Badut>',
    '\U0001f63a': '<Wajah Kucing Tersenyum Mulut Terbuka>',
    '\U0001f638': '<Wajah Kucing Menyeringai Mata Tersenyum>',
    '\U0001f63b': '<Wajah Kucing Tersenyum Mata Hati>',
    '\U0001f63c': '<Wajah Kucing Tersenyum Kecut>',
    '\U0001f63d': '<Wajah Kucing Mencium Mata Tertutup>',
    '\u2708': '<Pesawat Terbang>',
    '\U0001f47c': '<Bayi Malaikat>',
    '\U0001f4aa': '<Otot Lengan Kuat>',
    '\U0001f590': '<Tangan Terbuka Jari Merenggang>',
    '\U0001f91d': '<Berjabat Tangan>',
    '\U0001f48b': '<Tanda Kecupan Ciuman>',
    '\U0001f49e': '<Hati Berputar>',
    '\U0001f49d': '<Hati Dengan Pita Hadiah>',
    '\U0001f48e': '<Batu Permata Berlian>',
    '\U0001f425': '<Anak Ayam Menghadap Depan>',
    '\U0001f490': '<Buket Bunga>',
    '\U0001f339': '<Bunga Mawar>',
    '\U0001f31b': '<Bulan Sabit Awal Berwajah>',
    '\U0001f31c': '<Bulan Sabit Akhir Berwajah>',
    '\U0001f31d': '<Bulan Purnama Berwajah>',
    '\U0001f31e': '<Matahari Berwajah>',
    '\u2b50': '<Bintang Putih Sedang>',
    '\U0001f308': '<Pelangi>',
    '\U0001f380': '<Pita>',
    '\U0001f381': '<Kado Hadiah>',
    '\U0001f4a1': '<Bohlam Lampu Ide>',
    '\U0001f4c8': '<Grafik Meningkat>',
    '\U0001f4af': '<Nilai Seratus Sempurna>',
    '\U0001f197': '<Tombol Ok>',
}

emoji_to_tag_dict = {emoji: tag.lower() for emoji, tag in raw_emoji_to_tag.items()}
custom_tokens = list(emoji_to_tag_dict.values())
tokenizer.add_tokens(custom_tokens)

proposed_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    classifier_dropout=0.3,
    use_safetensors=True,
)
proposed_model.resize_token_embeddings(len(tokenizer))

print(f'Berhasil mendaftarkan {len(custom_tokens)} token emoji custom ke XLM-RoBERTa tokenizer!')
print(f'Model: XLMRobertaForSequenceClassification | num_labels=3 | classifier_dropout=0.3')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution 

Berhasil mendaftarkan 153 token emoji custom ke XLM-RoBERTa tokenizer!
Model: XLMRobertaForSequenceClassification | num_labels=3 | classifier_dropout=0.3


In [3]:
# Path dataset lokal (merged dari positif.csv + negatif.csv + netral.csv)
BASE_DIR = '/Users/hanaazizah/Developer/sentiment-analysis-emoji'
DATASET_PATH = os.path.join(BASE_DIR, 'dataset_final.csv')
CHECKPOINT_PATH = os.path.join(BASE_DIR, 'best_model.pt')
HISTORY_PATH = os.path.join(BASE_DIR, 'history_validation.csv')

df_proses = pd.read_csv(DATASET_PATH)

# Hapus kolom yang tidak diperlukan
for col in ['tweet emoji', 'Unnamed: 0']:
    if col in df_proses.columns:
        df_proses = df_proses.drop(columns=[col])

# Stopword: aman dihapus (bukan negasi/intensifier/partikel sentimen)
# JANGAN hapus: tidak, bukan, belum, jangan, tak, ga, gak, nggak (negasi)
# JANGAN hapus: sangat, banget, sekali, amat, terlalu, parah (intensifier)
# JANGAN hapus: sih, deh, dong, nih, lho, kan (partikel tone/sentimen)
STOPWORDS = {
    'yang', 'di', 'ke', 'dari', 'dengan', 'untuk', 'ini', 'itu',
    'oleh', 'pada', 'atau', 'dan', 'juga', 'telah', 'ada', 'adalah',
    'pun', 'pula', 'saja', 'lah', 'kah', 'si', 'sang', 'para',
    'maka', 'agar', 'supaya', 'karena', 'sebab', 'jika', 'kalau',
    'bila', 'apabila', 'walaupun', 'meskipun', 'setelah', 'sesudah',
    'sebelum', 'ketika', 'saat', 'waktu', 'seperti', 'antara',
    'dalam', 'semua', 'setiap', 'tiap', 'masing', 'bisa', 'dapat',
    'boleh', 'perlu', 'sedang', 'kami', 'kita', 'mereka', 'dia',
    'ia', 'anda', 'saya', 'aku', 'kamu', 'kalian', 'nya', 'mu', 'ku',
}

def clean_preprocessing(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\s*\S+|www\.\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-z\s<>]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Stopword removal: pisahkan tag emoji <...> agar tidak terganggu
    parts = re.split(r'(<[^>]*>)', text)
    filtered = []
    for part in parts:
        if re.match(r'<[^>]*>', part):
            filtered.append(part)
        else:
            tokens = [t for t in part.split() if t not in STOPWORDS]
            filtered.append(' '.join(tokens))
    text = re.sub(r'\s+', ' ', ' '.join(filtered)).strip()
    return text

df_proses['tweet_clean'] = df_proses['tweet_with_tags'].apply(clean_preprocessing)

# Filter teks terlalu pendek setelah cleaning (< 2 token)
sebelum = len(df_proses)
df_proses = df_proses[df_proses['tweet_clean'].str.split().str.len() >= 2].reset_index(drop=True)
print(f'Filter teks pendek: {sebelum} → {len(df_proses)} baris (hapus {sebelum - len(df_proses)})')
print(f'\nTotal data: {len(df_proses)} baris')
print(df_proses[['tweet_with_tags', 'tweet_clean', 'sentimen llm']].head())

Filter teks pendek: 18720 → 18720 baris (hapus 0)

Total data: 18720 baris
                                     tweet_with_tags  \
0  Asalamualaikum salam kenal saya dari Sabang sa...   
1  Baju ungu seperti belatung nangka lucu anet <W...   
2  Maybe orang yang snap tu terkejut <Wajah Berte...   
3  Napa siiiii..unjungnya gitu?sumpah gw nyesel n...   
4  Ayah ibu aku minta maaf ya kalau aku sering me...   

                                         tweet_clean sentimen llm  
0  asalamualaikum salam kenal sabang sampai merau...       netral  
1  baju ungu belatung nangka lucu anet <wajah men...      negatif  
2  maybe orang snap tu terkejut <wajah berteriak ...       netral  
3  napa siiiiiunjungnya gitusumpah gw nyesel nont...      positif  
4  ayah ibu minta maaf ya sering membuat marah ga...      positif  


### 2. Inspeksi Data

In [4]:
print('Jumlah data awal per kelas sentimen:')
print(df_proses['sentimen llm'].value_counts())
print(f'\nTotal data: {len(df_proses)} baris')

Jumlah data awal per kelas sentimen:
sentimen llm
negatif    7969
netral     6105
positif    4646
Name: count, dtype: int64

Total data: 18720 baris


In [5]:
label_mapping = {'negatif': 0, 'netral': 1, 'positif': 2}
df_proses['label'] = df_proses['sentimen llm'].map(label_mapping)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=df_proses['label'].values
)
class_weights_tensor = torch.FloatTensor(class_weights)

print('Distribusi kelas asli (semua data dipakai, tidak undersample):')
print(df_proses['sentimen llm'].value_counts())
print(f'\nTotal data: {len(df_proses)} baris')
print(f'\nClass weights → negatif: {class_weights[0]:.4f} | netral: {class_weights[1]:.4f} | positif: {class_weights[2]:.4f}')

Distribusi kelas asli (semua data dipakai, tidak undersample):
sentimen llm
negatif    7969
netral     6105
positif    4646
Name: count, dtype: int64

Total data: 18720 baris

Class weights → negatif: 0.7830 | netral: 1.0221 | positif: 1.3431


In [6]:
train_df, val_df = train_test_split(
    df_proses,
    test_size=0.2,
    random_state=42,
    stratify=df_proses['label']
)

print(f'Data Train: {len(train_df)} baris')
print(f'Data Val  : {len(val_df)} baris')
print('\nDistribusi kelas di Data Train:')
print(train_df['sentimen llm'].value_counts())
print('\nDistribusi kelas di Data Val:')
print(val_df['sentimen llm'].value_counts())

Data Train: 14976 baris
Data Val  : 3744 baris

Distribusi kelas di Data Train:
sentimen llm
negatif    6375
netral     4884
positif    3717
Name: count, dtype: int64

Distribusi kelas di Data Val:
sentimen llm
negatif    1594
netral     1221
positif     929
Name: count, dtype: int64


### 3. Setup Model

In [7]:
class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['tweet_clean']
        label = row['label']

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [8]:
BATCH_SIZE = 4

train_loader = DataLoader(SentimentDataset(train_df, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SentimentDataset(val_df,   tokenizer), batch_size=BATCH_SIZE, shuffle=False)

print('DataLoader berhasil diinisialisasi!')
sample_batch = next(iter(train_loader))
print(f'Ukuran tensor Input IDs per batch : {sample_batch["input_ids"].shape}')
print(f'Ukuran tensor Labels per batch    : {sample_batch["label"].shape}')

DataLoader berhasil diinisialisasi!
Ukuran tensor Input IDs per batch : torch.Size([4, 256])
Ukuran tensor Labels per batch    : torch.Size([4])


### 4. Setup Parameter

In [9]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Device yang digunakan: {device}')

proposed_model = proposed_model.to(device)

# Full fine-tuning: unfreeze SEMUA layer
for param in proposed_model.parameters():
    param.requires_grad = True

# Differential LR: backbone kecil agar tidak merusak pretrained weights
bert_params = list(proposed_model.roberta.parameters())
head_params = list(proposed_model.classifier.parameters())

optimizer = torch.optim.AdamW([
    {'params': bert_params, 'lr': 2e-6},
    {'params': head_params,  'lr': 1e-5},
], weight_decay=0.02)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor.to(device),
    label_smoothing=0.05
)

trainable = sum(p.numel() for p in proposed_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in proposed_model.parameters())
print(f'Full fine-tuning: semua {trainable:,} / {total:,} params di-unfreeze ({100*trainable/total:.1f}%)')
print('Differential LR  → BERT backbone: 2e-6 | Classifier head: 1e-5')
print('weight_decay: 0.02 | label_smoothing: 0.05 | classifier_dropout: 0.3 | max_len: 256')

Device yang digunakan: mps
Full fine-tuning: semua 278,163,459 / 278,163,459 params di-unfreeze (100.0%)
Differential LR  → BERT backbone: 2e-6 | Classifier head: 1e-5
weight_decay: 0.02 | label_smoothing: 0.05 | classifier_dropout: 0.3 | max_len: 256


### 5. Training

In [ ]:
NUM_EPOCHS = 5
PATIENCE   = 2

scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print('Memulai Training & Validasi...')
print('-' * 75)

history_records = []
best_val_f1     = 0.0
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    # ── TRAINING ──
    proposed_model.train()
    total_train_loss = 0
    all_train_preds, all_train_labels = [], []

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = proposed_model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits

        loss = criterion(logits, labels)
        total_train_loss += loss.item()

        _, preds = torch.max(logits, dim=1)
        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(proposed_model.parameters(), max_norm=1.0)
        optimizer.step()

    scheduler.step()
    current_lr = scheduler.get_last_lr()

    avg_train_loss = total_train_loss / len(train_loader)
    train_acc = accuracy_score(all_train_labels, all_train_preds)
    train_prec, train_rec, train_f1, _ = precision_recall_fscore_support(
        all_train_labels, all_train_preds, average='macro', zero_division=0
    )

    # ── VALIDASI ──
    proposed_model.eval()
    total_val_loss = 0
    all_val_preds, all_val_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = proposed_model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits

            loss = criterion(logits, labels)
            total_val_loss += loss.item()

            _, preds = torch.max(logits, dim=1)
            all_val_preds.extend(preds.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    val_acc = accuracy_score(all_val_labels, all_val_preds)
    val_prec, val_rec, val_f1, _ = precision_recall_fscore_support(
        all_val_labels, all_val_preds, average='macro', zero_division=0
    )

    elapsed = time.time() - start_time

    # ── Checkpoint & Early Stopping ──
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': proposed_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
        }, CHECKPOINT_PATH)
        marker = ' ← best (saved)'
    else:
        epochs_no_improve += 1
        marker = f' (no improve {epochs_no_improve}/{PATIENCE})'

    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] | {elapsed:.2f}s | LR: {current_lr[0]:.2e}')
    print(f'  [TRAIN] Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}')
    print(f'  [VALID] Loss: {avg_val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}{marker}')
    print('-' * 75)

    history_records.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss, 'train_accuracy': train_acc,
        'train_precision': train_prec, 'train_recall': train_rec, 'train_f1_score': train_f1,
        'val_loss': avg_val_loss, 'val_accuracy': val_acc,
        'val_precision': val_prec, 'val_recall': val_rec, 'val_f1_score': val_f1
    })

    if epochs_no_improve >= PATIENCE:
        print(f'\nEarly stopping triggered! Val F1 tidak improve selama {PATIENCE} epoch berturut-turut.')
        break

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
proposed_model.load_state_dict(checkpoint['model_state_dict'])
print(f'\nTraining selesai! Best Val F1: {best_val_f1:.4f} (epoch {checkpoint["epoch"]})')
print(f'Best model di-load dari: \'{CHECKPOINT_PATH}\'')

df_history = pd.DataFrame(history_records)
df_history.to_csv(HISTORY_PATH, index=False)
print(f'History disimpan di: \'{HISTORY_PATH}\'')

Memulai Training & Validasi...
---------------------------------------------------------------------------
